# 03 — Frozen DINOv2 + Ridge Regression (Milestone 3)

**Goal:** Build and validate our first image-based machine learning model for soil grain-size prediction.

Pipeline:
1. Map all 162 photographs (127 train / 35 test) to canonical soil sample IDs and compute effective pixels-per-millimeter (`effective_ppm`) from `ppm_updated.csv` + EXIF-oriented dimensions.
2. Verify exact competition EMD normalization (`[0, 500]`, equivalent to `100 * Wasserstein-1` on `log10(DIAMETERS)`).
3. Preprocess each image with EXIF orientation correction and multi-scale physical-square center crops (`90 mm` and `150 mm`, resized to `224x224`) to normalize across the 4x camera scale shift between training (`~4.55–4.60 px/mm`) and test (`13.94–19.53 px/mm`) phones.
4. Extract frozen DINOv2 (`vit_small_patch14_dinov2.lvd142m`) CLS + mean-patch embeddings and aggregate at the soil-sample level.
5. Evaluate Ridge regression with strictly leakage-free nested leave-one-sample-out cross-validation (LOOCV) and compare against the `89.888867` mean-curve baseline.
6. Generate and validate `submissions/submission.csv` if the model improves over the baseline.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance

from src.baseline import loocv_mean_baseline, read_labels
from src.dataset import build_image_manifest, resolve_data_dir
from src.features import DEFAULT_DINOV2_MODEL, DINOv2FeatureExtractor, aggregate_sample_features
from src.metric import DIAMETERS, LOG_WEIGHTS, TARGET_COLUMNS, emd_per_sample, validate_curves
from src.model import (
    DEFAULT_ALPHA_GRID,
    evaluate_nested_loocv_ridge,
    fit_predict_ridge,
    select_best_alpha_loocv,
    write_model_submission,
)

DATA_DIR = resolve_data_dir(repo_root=ROOT)
LABELS_PATH = DATA_DIR / "Training_labels_updated.csv"
TEMPLATE_PATH = DATA_DIR / "sample_submission.csv"
print("Dataset directory:", DATA_DIR)

## 1. Verify image-to-sample mapping and camera scale (PPM)

In [ ]:
labels_df = read_labels(LABELS_PATH)
template_df = pd.read_csv(TEMPLATE_PATH)

train_manifest = build_image_manifest(DATA_DIR, "train")
test_manifest = build_image_manifest(DATA_DIR, "test")

print(f"Train: {len(train_manifest)} images across {train_manifest['sample_id'].nunique()} samples")
print(f"Test:  {len(test_manifest)} images across {test_manifest['sample_id'].nunique()} samples")
display(
    pd.concat([train_manifest, test_manifest], ignore_index=True)
    .groupby(["split", "phone"])[["effective_ppm", "fov_short_mm", "fov_long_mm"]]
    .agg(["count", "mean"])
)

## 2. Verify competition EMD normalization (`[0, 500]`)

In [ ]:
finest = np.array([100.0] * 11)
coarsest = np.array([0.0] * 10 + [100.0])
max_emd = emd_per_sample(finest, coarsest)[0]
assert np.isclose(max_emd, 500.0)

y_all = labels_df[TARGET_COLUMNS].to_numpy(dtype=float)
pdf_0 = np.diff(np.concatenate([[0.0], y_all[0]])) / 100.0
pdf_1 = np.diff(np.concatenate([[0.0], y_all[1]])) / 100.0
w1 = 100.0 * wasserstein_distance(np.log10(DIAMETERS), np.log10(DIAMETERS), u_weights=pdf_0, v_weights=pdf_1)
emd_01 = emd_per_sample(y_all[0], y_all[1])[0]
assert np.isclose(emd_01, w1, rtol=1e-6)

print(f"Max EMD (range [0, 500]): {max_emd:.1f}")
print(f"Sum of log10 weights:     {LOG_WEIGHTS.sum():.1f}")
print(f"Wasserstein-1 check:      emd={emd_01:.6f} vs scipy={w1:.6f} (exact match)")

## 3. Extract frozen DINOv2 embeddings and aggregate by soil sample

In [ ]:
extractor = DINOv2FeatureExtractor(
    model_name=DEFAULT_DINOV2_MODEL,
    image_size=224,
    crop_mms=(90.0, 150.0),
    pool="cls_patch_mean",
)

train_img_feats = extractor.extract_manifest_features(train_manifest, batch_size=16)
test_img_feats = extractor.extract_manifest_features(test_manifest, batch_size=16)

train_ids = labels_df["sample_id"].tolist()
test_ids = template_df["sample_id"].tolist()

X_train = aggregate_sample_features(train_img_feats, train_manifest["sample_id"].tolist(), train_ids, agg="mean")
X_test = aggregate_sample_features(test_img_feats, test_manifest["sample_id"].tolist(), test_ids, agg="mean")
y_train = labels_df[TARGET_COLUMNS].to_numpy(dtype=float)

print("X_train:", X_train.shape, "| X_test:", X_test.shape, "| y_train:", y_train.shape)

## 4. Leakage-free LOOCV Ridge evaluation vs mean baseline

In [ ]:
baseline_emd = loocv_mean_baseline(labels_df)

# Compute per-sample baseline EMD for side-by-side comparison
baseline_preds = np.vstack([
    y_train[np.arange(len(y_train)) != i].mean(axis=0)
    for i in range(len(y_train))
])
baseline_per_sample = emd_per_sample(y_train, baseline_preds)

nested_res = evaluate_nested_loocv_ridge(X_train, y_train, alphas=DEFAULT_ALPHA_GRID)
best_alpha, alpha_grid_df = select_best_alpha_loocv(X_train, y_train, alphas=DEFAULT_ALPHA_GRID)

improvement = baseline_emd - nested_res.mean_emd
rel_improvement = 100.0 * improvement / baseline_emd

print(f"Mean baseline LOOCV EMD:        {baseline_emd:.6f}")
print(f"DINOv2 + Ridge Nested LOOCV:    {nested_res.mean_emd:.6f}")
print(f"Improvement over baseline:      {improvement:+.6f} ({rel_improvement:.2f}% reduction)")
print(f"Best fixed alpha on full LOOCV: {best_alpha} (EMD={alpha_grid_df.loc[0, 'loocv_emd']:.6f})")
display(alpha_grid_df.head(6))

## 5. Per-sample diagnostics and curve comparisons

In [ ]:
comparison = pd.DataFrame({
    "sample_id": train_ids,
    "baseline_emd": baseline_per_sample,
    "dinov2_ridge_emd": nested_res.per_sample_emd,
    "delta": nested_res.per_sample_emd - baseline_per_sample,
}).sort_values("dinov2_ridge_emd")

print(f"Samples improved over baseline: {int((comparison['delta'] < 0).sum())} / {len(comparison)}")
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: per-sample EMD bar comparison
ax = axes[0]
order = np.argsort(baseline_per_sample)
y_pos = np.arange(len(train_ids))
ax.barh(y_pos - 0.18, baseline_per_sample[order], height=0.35, alpha=0.6, label="Mean baseline")
ax.barh(y_pos + 0.18, nested_res.per_sample_emd[order], height=0.35, alpha=0.85, label="DINOv2 + Ridge")
ax.set_yticks(y_pos)
ax.set_yticklabels([train_ids[i] for i in order], fontsize=7)
ax.set_xlabel("Sample EMD (lower is better)")
ax.set_title("Leave-One-Sample-Out EMD by Soil Sample")
ax.legend()
ax.grid(axis="x", alpha=0.25)

# Right: example predicted vs true curves (median + hardest + easiest)
ax = axes[1]
ranked = np.argsort(nested_res.per_sample_emd)
for idx, tag in [(ranked[0], "Best"), (ranked[len(ranked) // 2], "Median"), (ranked[-1], "Hardest")]:
    line, = ax.plot(DIAMETERS, y_train[idx], linestyle="-", linewidth=2, label=f"{tag} true ({train_ids[idx]})")
    ax.plot(DIAMETERS, nested_res.predictions[idx], linestyle="--", color=line.get_color(), label=f"{tag} pred (EMD={nested_res.per_sample_emd[idx]:.1f})")
ax.set_xscale("log")
ax.set(xlabel="Grain diameter (mm)", ylabel="Cumulative passing mass (%)", ylim=(0, 100), title="Predicted vs True Grading Curves")
ax.grid(alpha=0.25)
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 6. Generate and validate `submissions/submission.csv`

In [ ]:
if nested_res.mean_emd < baseline_emd:
    test_preds = fit_predict_ridge(X_train, y_train, X_test, alpha=best_alpha)
    output_path = ROOT / "submissions" / "submission.csv"
    sub_df = write_model_submission(test_ids, test_preds, TEMPLATE_PATH, output_path)
    assert list(sub_df.columns) == list(template_df.columns)
    assert (sub_df["sample_id"].to_numpy() == template_df["sample_id"].to_numpy()).all()
    assert not sub_df.isna().any().any()
    validate_curves(sub_df[TARGET_COLUMNS].to_numpy(dtype=float))
    print(f"Saved and verified {output_path} ({sub_df.shape[0]} test samples, alpha={best_alpha})")
    display(sub_df)
else:
    print("Model did not improve over baseline; submission not generated.")